# Intraday Trader — Research Notebook
Exploratory analysis: features, labels, model diagnostics.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from data.data_loader import YFinanceLoader
from data.data_cleaning import DataCleaner
from features.feature_engineering import FeatureEngineer
from features.feature_selection import FeatureSelector
from labeling.barriers import select_label
from models.ensemble import EnsembleModel
%matplotlib inline

## 1. Load data

In [ ]:
SYMBOL = 'RELIANCE.NS'
loader  = YFinanceLoader()
cleaner = DataCleaner()
eng     = FeatureEngineer()

raw   = loader.fetch(SYMBOL)
df    = cleaner.clean(raw, SYMBOL)
bench = loader.fetch_benchmark()
bench = cleaner.clean(bench, 'NIFTY50') if not bench.empty else pd.DataFrame()
print(df.shape, df.index[0], df.index[-1])

## 2. Feature engineering

In [ ]:
bench_in = pd.DataFrame()
if not bench.empty:
    bi = bench.reindex(df.index, method='ffill').copy()
    bi['volume'] = bi.get('volume', pd.Series(1_000_000, index=bi.index))
    bench_in = bi

X = eng.build(df, benchmark=bench_in)
print('Features:', X.shape)
X.describe().T.sort_values('std', ascending=False).head(20)

## 3. Labels

In [ ]:
atr   = X['atr']
close = df['close'].reindex(X.index)
y     = select_label(close, atr)
y     = y.dropna().astype(int)
print('Pos rate:', y.mean().round(3))
y.value_counts().plot(kind='bar', title='Label Distribution')

## 4. Feature selection

In [ ]:
sel   = FeatureSelector(top_k=30)
X_sel = sel.fit_transform(X.loc[y.index], y)
print('Selected:', sel.selected)

## 5. Walk-forward model eval

In [ ]:
model = EnsembleModel()
folds = model.walk_forward(X_sel, y)
import pandas as pd
pd.DataFrame(folds)[['fold','auc','f1','win_rate','threshold']]

## 6. Feature correlation heatmap

In [ ]:
import seaborn as sns
corr = X_sel.corr()
plt.figure(figsize=(14, 12))
sns.heatmap(corr, cmap='RdBu_r', center=0, vmin=-1, vmax=1, linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()